# Constrained log-likelihood scoring — fine-tuned VLM, matched to OpenCLIP's protocol

In [ ]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [ ]:
import json
import re
import numpy as np
import pandas as pd
from pathlib import Path

RESULTS_PATH = "/content/drive/MyDrive/Surgical-VLM/results/VLM Results/visionllm_predictions.csv"
CHECKPOINT = "/content/drive/MyDrive/Surgical-VLM/checkpoints/best_qwen_vl_final"

CANONICAL_TASKS = [
    "Instrument Recognition",
    "Action Recognition",
    "Tissue and Organ Recognition",
    "Phase Recognition",
]

eval_df = pd.read_csv(RESULTS_PATH)
eval_df = eval_df[eval_df["task"].isin(CANONICAL_TASKS)].reset_index(drop=True)
print(eval_df.shape)
eval_df["task"].value_counts()


(3200, 5)


,count
task,
Action Recognition,800
Instrument Recognition,800
Phase Recognition,800
Tissue and Organ Recognition,800


## Canonical label vocabulary

Same CholecT50 ontology used throughout this project. If a synonym map was
defined in `15__Modified_zero-shot_eval.ipynb` (`CANDIDATE_SYNONYMS`), merge
it in here for consistency with the corrected zero-shot/fine-tuned parsing —
not reproduced here since it wasn\'t available at notebook-build time.

In [ ]:
INSTRUMENTS = ["irrigator", "bipolar", "grasper", "clipper", "hook", "scissors"]
ACTIONS = ["grasp", "retract", "dissect", "coagulate", "clip", "cut",
           "aspirate", "irrigate", "pack", "null_verb"]
TARGETS = ["blood_vessel", "cystic_plate", "gut", "cystic_pedicle", "peritoneum",
           "null_target", "adhesion", "gallbladder", "liver", "fluid", "omentum",
           "specimen_bag", "cystic_artery", "abdominal_wall_cavity", "cystic_duct"]
PHASES = ["Preparation", "Calot Triangle Dissection", "Clipping Cutting",
          "Gallbladder Dissection", "Gallbladder Packaging",
          "Cleaning Coagulation", "Gallbladder Retraction"]

CANONICAL_POOLS = {
    "Instrument Recognition": INSTRUMENTS,
    "Action Recognition": ACTIONS,
    "Tissue and Organ Recognition": TARGETS,
    "Phase Recognition": PHASES,
}

def _vocab_variants(term):
    t = term.lower()
    return {t, t.replace("_", " "), t.replace(" ", "_")}

def extract_ground_truth_labels(text, vocab):
    text = str(text).lower()
    found = []
    for term in vocab:
        canonical = term.lower().replace(" ", "_")
        for variant in _vocab_variants(term):
            if re.search(r"\b" + re.escape(variant) + r"\b", text):
                found.append(canonical)
                break
    return found

eval_df["ground_truth_labels"] = eval_df.apply(
    lambda row: extract_ground_truth_labels(row["ground_truth"], CANONICAL_POOLS[row["task"]]),
    axis=1,
)
print("Rows with empty extracted ground truth:", (eval_df["ground_truth_labels"].apply(len) == 0).sum())


Rows with empty extracted ground truth: 1


## Load the fine-tuned model


In [ ]:
!pip install --upgrade "torchao>=0.16.0" transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 125.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 152.6 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.13.1
    Uninstalling transformers-5.13.1:
      Successfully uninstalled transformers-5.13.1


In [ ]:
import torch
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration
from peft import PeftModel

MODEL_NAME = "Qwen/Qwen2.5-VL-3B-Instruct"

processor = AutoProcessor.from_pretrained(
    MODEL_NAME,
    min_pixels=256 * 28 * 28,
    max_pixels=1024 * 28 * 28,
)

base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model = PeftModel.from_pretrained(base_model, CHECKPOINT)
model = model.merge_and_unload()
model.eval()

processor.tokenizer.padding_side = "right"


preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/5.70k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/65.4k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

In [ ]:
import time, shutil, subprocess
from pathlib import Path

CONFIG = Path("/content/drive/MyDrive/Surgical-VLM/configs/config.json")

with open(CONFIG) as f:
    config = json.load(f)

DRIVE_CHOLECT50_ARCHIVE = Path("/content/drive/MyDrive/Surgical-VLM/data/CholecT50_raw.zip")  # adjust extension if .tar.gz
LOCAL_ARCHIVE_COPY = Path("/content/CholecT50_raw" + DRIVE_CHOLECT50_ARCHIVE.suffix)
LOCAL_CHOLECT50_DIR = Path("/content/CholecT50")

assert DRIVE_CHOLECT50_ARCHIVE.exists(), (
    f"Expected archive not found at {DRIVE_CHOLECT50_ARCHIVE}. "
    "Upload it to Drive first (see markdown above) before running this cell."
)


print("Copying archive to local disk..")
t0 = time.time()
shutil.copy2(DRIVE_CHOLECT50_ARCHIVE, LOCAL_ARCHIVE_COPY)
print(f"Copy done in {time.time()-t0:.1f}s")

LOCAL_CHOLECT50_DIR.mkdir(parents=True, exist_ok=True)

print("Extracting locally...")
t0 = time.time()
if LOCAL_ARCHIVE_COPY.suffix == ".zip":
    subprocess.run(["unzip", "-q", str(LOCAL_ARCHIVE_COPY), "-d", str(LOCAL_CHOLECT50_DIR)], check=True)
else:
    subprocess.run(["tar", "-xzf", str(LOCAL_ARCHIVE_COPY), "-C", str(LOCAL_CHOLECT50_DIR)], check=True)
print(f"Extract done in {time.time()-t0:.1f}s")

n_files = sum(1 for _ in LOCAL_CHOLECT50_DIR.rglob("*") if _.is_file())
print(f"\n{n_files:,} files extracted to {LOCAL_CHOLECT50_DIR}")

# Free the local zip copy now that it's extracted -- no need to keep both
LOCAL_ARCHIVE_COPY.unlink()



Copying archive to local disk..
Copy done in 1222.7s
Extracting locally...
Extract done in 633.1s

100,918 files extracted to /content/CholecT50


## Core scoring function

In [ ]:
from PIL import Image

def score_candidates(image, question, candidates):
    """Returns a list of summed log-probabilities, one per candidate,
    plus the raw sliced candidate token ids (for the sanity check)."""
    messages = [{"role": "user", "content": [
        {"type": "image", "image": image},
        {"type": "text", "text": question},
    ]}]
    prompt_text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # true prompt length, measured the same way as the training collate_fn:
    # run the processor on the prompt alone (with image) and read the
    # unpadded attention mask length, NOT a plain-tokenizer count
    prompt_only_inputs = processor(text=[prompt_text], images=[image], return_tensors="pt")
    prompt_len = prompt_only_inputs["input_ids"].shape[1]

    full_texts = [prompt_text + c for c in candidates]
    inputs = processor(text=full_texts, images=[image] * len(candidates), return_tensors="pt", padding=True).to(model.device)

    with torch.no_grad():
        outputs = model(**inputs)
    log_probs = torch.log_softmax(outputs.logits.float(), dim=-1)

    scores = []
    sliced_token_ids = []
    for i in range(len(candidates)):
        input_ids_i = inputs["input_ids"][i]
        total_len = int(inputs["attention_mask"][i].sum())
        cand_ids = input_ids_i[prompt_len:total_len]
        sliced_token_ids.append(cand_ids)

        lp = 0.0
        for t in range(prompt_len, total_len):
            token_id = input_ids_i[t].item()
            lp += log_probs[i, t - 1, token_id].item()
        scores.append(lp)

    return scores, sliced_token_ids


## Sanity check

In [ ]:
sample = eval_df.iloc[0]
image = Image.open(sample["image_full_path"]).convert("RGB")
task = sample["task"]
candidates = CANONICAL_POOLS[task]

scores, sliced_ids = score_candidates(image, sample["question"], candidates)

print("Task:", task)
print("Ground truth:", sample["ground_truth"])
print("Ground truth labels (extracted):", sample["ground_truth_labels"])
print()
for cand, score, ids in zip(candidates, scores, sliced_ids):
    decoded = processor.tokenizer.decode(ids)
    print(f"  candidate={cand!r:20s} decoded_slice={decoded!r:25s} log-likelihood={score:.3f}")

print()
best_idx = int(np.argmax(scores))
print("Argmax candidate:", candidates[best_idx])
print("Correct (in ground truth set)?", candidates[best_idx] in sample["ground_truth_labels"])
print()
print("CHECK 1 — do the decoded_slice values above match their candidate strings (ignoring leading/trailing whitespace)?")
print("CHECK 2 — does the argmax candidate look like a reasonable guess given the ground truth?")
print("If either check fails, STOP and debug prompt_len / slicing before running the full loop below.")


Task: Action Recognition
Ground truth: The grasper is performing a retract action in this surgery image.
Ground truth labels (extracted): ['retract']

  candidate='grasp'              decoded_slice='grasp'                   log-likelihood=-22.511
  candidate='retract'            decoded_slice='retract'                 log-likelihood=-19.750
  candidate='dissect'            decoded_slice='dissect'                 log-likelihood=-21.751
  candidate='coagulate'          decoded_slice='coagulate'               log-likelihood=-25.752
  candidate='clip'               decoded_slice='clip'                    log-likelihood=-26.812
  candidate='cut'                decoded_slice='cut'                     log-likelihood=-29.562
  candidate='aspirate'           decoded_slice='aspirate'                log-likelihood=-28.377
  candidate='irrigate'           decoded_slice='irrigate'                log-likelihood=-24.802
  candidate='pack'               decoded_slice='pack'                    log-like

## Full run over all four tasks

In [ ]:
from tqdm.auto import tqdm

SAVE_PATH = "/content/drive/MyDrive/Surgical-VLM/results/vlm_loglik_canonical_results.csv"

results = []
for i, row in tqdm(eval_df.iterrows(), total=len(eval_df)):
    task = row["task"]
    candidates = CANONICAL_POOLS[task]
    try:
        image = Image.open(row["image_full_path"]).convert("RGB")
        scores, _ = score_candidates(image, row["question"], candidates)
        best_idx = int(np.argmax(scores))
        predicted_label = candidates[best_idx]
        top1_correct = predicted_label in row["ground_truth_labels"]
    except Exception as e:
        print(f"Row {i} ({task}) failed: {e}")
        predicted_label = None
        top1_correct = False

    results.append({
        "original_index": i,
        "task": task,
        "ground_truth_text": row["ground_truth"],
        "predicted_label": predicted_label,
        "top1_correct": top1_correct,
        "candidate_count": len(candidates),
    })

    if i % 50 == 0:
        pd.DataFrame(results).to_csv(SAVE_PATH, index=False)

loglik_results = pd.DataFrame(results)
loglik_results.to_csv(SAVE_PATH, index=False)
print(f"Saved {len(loglik_results)} rows to {SAVE_PATH}")


  0%|          | 0/3200 [00:00<?, ?it/s]

## Summary table

In [ ]:
summary_rows = []
for task in CANONICAL_TASKS:
    subset = loglik_results[loglik_results["task"] == task]
    accuracy = subset["top1_correct"].mean()
    candidate_count = len(CANONICAL_POOLS[task])
    chance = 1.0 / candidate_count
    summary_rows.append({
        "Task": task,
        "Candidate count": candidate_count,
        "N evaluated": len(subset),
        "Top-1 accuracy": accuracy,
        "Chance": chance,
        "Lift over chance": accuracy - chance,
    })

vlm_loglik_summary = pd.DataFrame(summary_rows)
vlm_loglik_summary.to_csv(
    "/content/drive/MyDrive/Surgical-VLM/results/vlm_loglik_canonical_summary.csv",
    index=False,
)
vlm_loglik_summary
